# DBSCAN CUDA C++/nvcc - Multi-EPS

This notebook follows the same style as the Gemini/CUDA notebook: RAPIDS/cuML baseline, CUDA C++ kernels, `%%writefile`, `nvcc`, and binary execution.

The idea comes from the attached MultiK paper (`30984-73-25314-1-10-20241023_260604_015357.pdf`): MultiK evaluates multiple values of `k` simultaneously to reuse input data reads and increase arithmetic intensity. Here, the analogous idea is Multi-EPS for DBSCAN: compute several `eps` values in one CUDA execution, reusing the same pairwise distance calculation.

Implemented EPS values:

- `eps_low = 0.75 * EPS_BASE`
- `eps_base = EPS_BASE`
- `eps_high = 1.25 * EPS_BASE`

For each EPS, the binary produces its own DBSCAN labels. The notebook compares each result with a separate cuML/sklearn baseline for the same EPS.


In [ ]:
import os
import re
import math
import time
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN as SklearnDBSCAN
from sklearn.datasets import make_blobs, make_moons, load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors

SEED = 42
np.random.seed(SEED)

HAS_CUML = False
cp = None
CuMLDBSCAN = None

try:
    import cupy as cp
    from cuml.cluster import DBSCAN as CuMLDBSCAN
    HAS_CUML = True
    print("RAPIDS/cuML available. Baseline: cuML.")
except Exception as exc:
    print("RAPIDS/cuML not available. Baseline fallback: sklearn CPU.")
    print("Reason:", repr(exc))

print("HAS_CUML:", HAS_CUML)


In [ ]:
def normalize_minmax(X):
    X = np.asarray(X, dtype=np.float32)
    mn = X.min(axis=0)
    mx = X.max(axis=0)
    denom = mx - mn
    denom[denom == 0.0] = 1.0
    return ((X - mn) / denom).astype(np.float32)


def estimate_eps(X, min_samples=8, quantile=0.90):
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(X)
    dists, _ = nn.kneighbors(X)
    return float(np.quantile(dists[:, -1], quantile))


def relabel_consecutive(labels):
    labels = np.asarray(labels, dtype=np.int32)
    out = np.full(labels.shape, -1, dtype=np.int32)
    valid = [int(v) for v in np.unique(labels) if int(v) != -1]
    for new, old in enumerate(sorted(valid)):
        out[labels == old] = new
    return out


def count_clusters(labels):
    labels = np.asarray(labels)
    values = set(labels.tolist())
    values.discard(-1)
    return len(values)


def noise_percent(labels):
    return 100.0 * float(np.mean(np.asarray(labels) == -1))


def run_baseline_dbscan(X, eps, min_samples):
    X = np.ascontiguousarray(X.astype(np.float32))
    start = time.time()
    if HAS_CUML:
        X_gpu = cp.asarray(X)
        model = CuMLDBSCAN(eps=float(eps), min_samples=int(min_samples))
        labels_gpu = model.fit_predict(X_gpu)
        cp.cuda.Stream.null.synchronize()
        labels = cp.asnumpy(labels_gpu).astype(np.int32)
        backend = "cuML"
    else:
        model = SklearnDBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=-1)
        labels = model.fit_predict(X).astype(np.int32)
        backend = "sklearn_cpu"
    elapsed = time.time() - start
    return relabel_consecutive(labels), elapsed, backend


def evaluate(version, eps_name, eps, labels, elapsed, baseline_labels, extra=None):
    extra = {} if extra is None else dict(extra)
    row = {
        "version": version,
        "eps_name": eps_name,
        "eps": float(eps),
        "time_s": float(elapsed),
        "clusters": count_clusters(labels),
        "noise_%": noise_percent(labels),
        "ARI_vs_baseline": adjusted_rand_score(baseline_labels, labels),
        "NMI_vs_baseline": normalized_mutual_info_score(baseline_labels, labels),
    }
    row.update(extra)
    return row


def print_table(df, title):
    print(title)
    print(df.where(pd.notna(df), "N/A").to_string(index=False))


def plot_2d(X, labels, title):
    if X.shape[1] == 2:
        X2 = X[:, :2]
    else:
        X2 = PCA(n_components=2, random_state=SEED).fit_transform(X)
    plt.figure(figsize=(6, 5))
    plt.scatter(X2[:, 0], X2[:, 1], c=labels, s=8, cmap="tab20", alpha=0.85, linewidths=0)
    plt.title(title)
    plt.xlabel("x1/PCA1")
    plt.ylabel("x2/PCA2")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Dataset selection
# ============================================================

RUN_MODE = "quick"  # "quick" or "presentation"
DATASET_NAME = "heterogeneous_blobs_2d"
MIN_SAMPLES = 8

if RUN_MODE == "quick":
    N_SAMPLES = 4000
else:
    N_SAMPLES = 12000


def make_controlled_dataset(name, n_samples, seed=SEED):
    rng = np.random.default_rng(seed)
    if name == "dense_blobs_2d":
        X, y = make_blobs(
            n_samples=n_samples,
            centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)],
            cluster_std=0.18,
            random_state=seed,
        )
    elif name == "heterogeneous_blobs_2d":
        n1 = int(0.45 * n_samples)
        n2 = int(0.35 * n_samples)
        n3 = n_samples - n1 - n2
        X1, _ = make_blobs(n_samples=n1, centers=[(-4, 0)], cluster_std=0.08, random_state=seed + 1)
        X2, _ = make_blobs(n_samples=n2, centers=[(0, 0)], cluster_std=0.25, random_state=seed + 2)
        X3, _ = make_blobs(n_samples=n3, centers=[(4, 0)], cluster_std=0.65, random_state=seed + 3)
        X = np.vstack([X1, X2, X3])
        y = np.concatenate([
            np.zeros(n1, dtype=np.int32),
            np.ones(n2, dtype=np.int32),
            np.full(n3, 2, dtype=np.int32),
        ])
    elif name == "dense_blobs_noise_2d":
        n_noise = int(0.20 * n_samples)
        n_blob = n_samples - n_noise
        X_blob, y_blob = make_blobs(
            n_samples=n_blob,
            centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)],
            cluster_std=0.20,
            random_state=seed,
        )
        noise = rng.uniform(low=-5.5, high=5.5, size=(n_noise, 2))
        X = np.vstack([X_blob, noise])
        y = np.concatenate([y_blob.astype(np.int32), np.full(n_noise, -1, dtype=np.int32)])
    elif name == "moons_2d":
        X, y = make_moons(n_samples=n_samples, noise=0.045, random_state=seed)
    elif name == "real_iris_4d":
        data = load_iris()
        X, y = data.data, data.target
    elif name == "real_wine_13d":
        data = load_wine()
        X, y = data.data, data.target
    elif name == "real_breast_cancer_30d":
        data = load_breast_cancer()
        X, y = data.data, data.target
    elif name == "real_digits_64d":
        data = load_digits()
        X, y = data.data, data.target
    else:
        raise ValueError(name)
    return normalize_minmax(X), np.asarray(y, dtype=np.int32)


X, y_true = make_controlled_dataset(DATASET_NAME, N_SAMPLES)
X = np.ascontiguousarray(X.astype(np.float32))
N, D = X.shape
EPS_BASE = estimate_eps(X, min_samples=MIN_SAMPLES, quantile=0.90)

# Multi-EPS values. Keep them sorted from low to high.
EPS_NAMES = ["eps_low", "eps_base", "eps_high"]
EPS_VALUES = np.array([0.75 * EPS_BASE, EPS_BASE, 1.25 * EPS_BASE], dtype=np.float32)

INPUT_BIN = "dbscan_multi_eps_input_f32.bin"
X.tofile(INPUT_BIN)

print("DATASET_NAME:", DATASET_NAME)
print("RUN_MODE:", RUN_MODE)
print("N, D:", N, D)
print("EPS_BASE:", EPS_BASE)
print("EPS_VALUES:", EPS_VALUES)
print("MIN_SAMPLES:", MIN_SAMPLES)
EPS0 = float(EPS_VALUES[0])
EPS1 = float(EPS_VALUES[1])
EPS2 = float(EPS_VALUES[2])

print("Saved:", INPUT_BIN)


In [ ]:
%%writefile dbscan_multi_eps.cu
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <math.h>
#include <fstream>
#include <vector>
#include <string>

#define NUM_EPS 3

#define CUDA_CHECK(call) do { \
    cudaError_t err = call; \
    if (err != cudaSuccess) { \
        printf("CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while(0)

__device__ int find_root_eps(int* parent, int base, int i) {
    int root = i;
    while (root != parent[base + root]) root = parent[base + root];
    return root;
}

__global__ void find_cores_multi_eps(
    const float* X,
    const float* eps_sq,
    int* is_core,
    int* neighbor_counts,
    int n,
    int d,
    int min_pts
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    int counts[NUM_EPS];
    #pragma unroll
    for (int e = 0; e < NUM_EPS; e++) counts[e] = 0;

    for (int j = 0; j < n; j++) {
        float dist_sq = 0.0f;
        for (int f = 0; f < d; f++) {
            float diff = X[i * d + f] - X[j * d + f];
            dist_sq += diff * diff;
            if (dist_sq > eps_sq[NUM_EPS - 1]) break;
        }

        #pragma unroll
        for (int e = 0; e < NUM_EPS; e++) {
            if (dist_sq <= eps_sq[e]) counts[e]++;
        }

        int all_core = 1;
        #pragma unroll
        for (int e = 0; e < NUM_EPS; e++) {
            if (counts[e] < min_pts) all_core = 0;
        }
        if (all_core) break;
    }

    #pragma unroll
    for (int e = 0; e < NUM_EPS; e++) {
        int idx = e * n + i;
        neighbor_counts[idx] = counts[e];
        is_core[idx] = (counts[e] >= min_pts) ? 1 : 0;
    }
}

__global__ void init_parents_multi_eps(const int* is_core, int* parent, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = NUM_EPS * n;
    if (idx >= total) return;
    int i = idx % n;
    parent[idx] = i;
}

__global__ void connect_cores_multi_eps(
    const float* X,
    const float* eps_sq,
    const int* is_core,
    int* parent,
    int n,
    int d
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    for (int j = i + 1; j < n; j++) {
        float dist_sq = 0.0f;
        for (int f = 0; f < d; f++) {
            float diff = X[i * d + f] - X[j * d + f];
            dist_sq += diff * diff;
            if (dist_sq > eps_sq[NUM_EPS - 1]) break;
        }

        #pragma unroll
        for (int e = 0; e < NUM_EPS; e++) {
            int base = e * n;
            if (dist_sq <= eps_sq[e] && is_core[base + i] && is_core[base + j]) {
                int root_i = find_root_eps(parent, base, i);
                int root_j = find_root_eps(parent, base, j);
                while (root_i != root_j) {
                    if (root_i < root_j) {
                        int old = atomicCAS(&parent[base + root_j], root_j, root_i);
                        if (old == root_j) break;
                        root_j = find_root_eps(parent, base, old);
                    } else {
                        int old = atomicCAS(&parent[base + root_i], root_i, root_j);
                        if (old == root_i) break;
                        root_i = find_root_eps(parent, base, old);
                    }
                }
            }
        }
    }
}

__global__ void flatten_multi_eps(int* parent, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = NUM_EPS * n;
    if (idx >= total) return;
    int e = idx / n;
    int i = idx % n;
    int base = e * n;
    int root = i;
    while (root != parent[base + root]) root = parent[base + root];
    parent[base + i] = root;
}

__global__ void assign_borders_multi_eps(
    const float* X,
    const float* eps_sq,
    const int* is_core,
    const int* parent,
    int* labels,
    int n,
    int d
) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = NUM_EPS * n;
    if (idx >= total) return;

    int e = idx / n;
    int i = idx % n;
    int base = e * n;

    if (is_core[base + i]) {
        labels[base + i] = parent[base + i];
        return;
    }

    int best_label = -1;
    float best_dist_sq = eps_sq[e];
    int found = 0;

    for (int j = 0; j < n; j++) {
        if (!is_core[base + j]) continue;

        float dist_sq = 0.0f;
        for (int f = 0; f < d; f++) {
            float diff = X[i * d + f] - X[j * d + f];
            dist_sq += diff * diff;
            if (dist_sq > eps_sq[e]) break;
        }

        if (dist_sq <= eps_sq[e] && (!found || dist_sq < best_dist_sq)) {
            found = 1;
            best_dist_sq = dist_sq;
            best_label = parent[base + j];
        }
    }
    labels[base + i] = best_label;
}

int main(int argc, char** argv) {
    if (argc < 10) {
        printf("Usage: %s input.bin output_labels.bin n d eps0 eps1 eps2 min_pts output_counts.bin\n", argv[0]);
        return 1;
    }

    std::string input_path = argv[1];
    std::string output_labels_path = argv[2];
    int n = atoi(argv[3]);
    int d = atoi(argv[4]);
    float h_eps[NUM_EPS];
    h_eps[0] = (float)atof(argv[5]);
    h_eps[1] = (float)atof(argv[6]);
    h_eps[2] = (float)atof(argv[7]);
    int min_pts = atoi(argv[8]);
    std::string output_counts_path = argv[9];

    float h_eps_sq[NUM_EPS];
    for (int e = 0; e < NUM_EPS; e++) h_eps_sq[e] = h_eps[e] * h_eps[e];

    size_t data_count = (size_t)n * (size_t)d;
    std::vector<float> h_X(data_count);
    std::ifstream in(input_path, std::ios::binary);
    if (!in) { printf("Could not open input file.\n"); return 1; }
    in.read((char*)h_X.data(), data_count * sizeof(float));
    in.close();

    float* d_X = nullptr;
    float* d_eps_sq = nullptr;
    int *d_is_core = nullptr, *d_parent = nullptr, *d_labels = nullptr, *d_neighbor_counts = nullptr;
    CUDA_CHECK(cudaMalloc(&d_X, data_count * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_eps_sq, NUM_EPS * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_is_core, NUM_EPS * n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_parent, NUM_EPS * n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_labels, NUM_EPS * n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_neighbor_counts, NUM_EPS * n * sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_X, h_X.data(), data_count * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_eps_sq, h_eps_sq, NUM_EPS * sizeof(float), cudaMemcpyHostToDevice));

    int threads = 256;
    int blocks_n = (n + threads - 1) / threads;
    int blocks_multi = (NUM_EPS * n + threads - 1) / threads;

    cudaEvent_t e0, e1, e2, e3, e4, e5;
    cudaEventCreate(&e0); cudaEventCreate(&e1); cudaEventCreate(&e2);
    cudaEventCreate(&e3); cudaEventCreate(&e4); cudaEventCreate(&e5);

    cudaEventRecord(e0);
    find_cores_multi_eps<<<blocks_n, threads>>>(d_X, d_eps_sq, d_is_core, d_neighbor_counts, n, d, min_pts);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEventRecord(e1);

    init_parents_multi_eps<<<blocks_multi, threads>>>(d_is_core, d_parent, n);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEventRecord(e2);

    connect_cores_multi_eps<<<blocks_n, threads>>>(d_X, d_eps_sq, d_is_core, d_parent, n, d);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEventRecord(e3);

    flatten_multi_eps<<<blocks_multi, threads>>>(d_parent, n);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEventRecord(e4);

    assign_borders_multi_eps<<<blocks_multi, threads>>>(d_X, d_eps_sq, d_is_core, d_parent, d_labels, n, d);
    CUDA_CHECK(cudaDeviceSynchronize());
    cudaEventRecord(e5);
    cudaEventSynchronize(e5);

    float ms_find=0, ms_init=0, ms_connect=0, ms_flatten=0, ms_assign=0;
    cudaEventElapsedTime(&ms_find, e0, e1);
    cudaEventElapsedTime(&ms_init, e1, e2);
    cudaEventElapsedTime(&ms_connect, e2, e3);
    cudaEventElapsedTime(&ms_flatten, e3, e4);
    cudaEventElapsedTime(&ms_assign, e4, e5);

    std::vector<int> h_labels(NUM_EPS * n);
    std::vector<int> h_counts(NUM_EPS * n);
    std::vector<int> h_core(NUM_EPS * n);
    CUDA_CHECK(cudaMemcpy(h_labels.data(), d_labels, NUM_EPS * n * sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_counts.data(), d_neighbor_counts, NUM_EPS * n * sizeof(int), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_core.data(), d_is_core, NUM_EPS * n * sizeof(int), cudaMemcpyDeviceToHost));

    std::ofstream out_labels(output_labels_path, std::ios::binary);
    out_labels.write((char*)h_labels.data(), NUM_EPS * n * sizeof(int));
    out_labels.close();

    std::ofstream out_counts(output_counts_path, std::ios::binary);
    out_counts.write((char*)h_counts.data(), NUM_EPS * n * sizeof(int));
    out_counts.close();

    printf("n=%d d=%d min_pts=%d\n", n, d, min_pts);
    for (int e = 0; e < NUM_EPS; e++) {
        int base = e * n;
        int n_core = 0;
        int n_noise = 0;
        for (int i = 0; i < n; i++) {
            n_core += h_core[base + i];
            if (h_labels[base + i] == -1) n_noise++;
        }
        printf("eps_index=%d eps=%f core_points=%d noise=%d\n", e, h_eps[e], n_core, n_noise);
    }
    printf("time_find=%.6f time_init=%.6f time_connect=%.6f time_flatten=%.6f time_assign=%.6f total=%.6f seconds\n",
           ms_find/1000.0f, ms_init/1000.0f, ms_connect/1000.0f, ms_flatten/1000.0f, ms_assign/1000.0f,
           (ms_find + ms_init + ms_connect + ms_flatten + ms_assign)/1000.0f);

    cudaFree(d_X); cudaFree(d_eps_sq); cudaFree(d_is_core); cudaFree(d_parent); cudaFree(d_labels); cudaFree(d_neighbor_counts);
    return 0;
}


In [ ]:
# Otimizacao de compilacao flag -O3 para velocidade maxima da CPU e GPU
!nvcc dbscan_multi_eps.cu -o dbscan_multi_eps -O3

# Executa o binario gerado no estilo do Colab original.
# O binario calcula eps_low, eps_base e eps_high na mesma execucao.
# A proxima celula roda novamente e monta a tabela comparando com os baselines separados.
!nvprof ./dbscan_multi_eps {INPUT_BIN} nvprof_multi_eps_labels.bin {N} {D} {EPS0} {EPS1} {EPS2} {MIN_SAMPLES} nvprof_multi_eps_counts.bin


In [ ]:
# ============================================================
# Baselines: run cuML/sklearn separately for each EPS
# ============================================================

baseline_labels_by_eps = {}
baseline_rows = []

for eps_name, eps_value in zip(EPS_NAMES, EPS_VALUES):
    labels, elapsed, backend = run_baseline_dbscan(X, float(eps_value), MIN_SAMPLES)
    baseline_labels_by_eps[eps_name] = labels
    baseline_rows.append({
        "version": f"baseline_{backend}",
        "eps_name": eps_name,
        "eps": float(eps_value),
        "time_s": elapsed,
        "clusters": count_clusters(labels),
        "noise_%": noise_percent(labels),
    })
    print(f"baseline {backend} {eps_name}: {elapsed:.4f}s")

df_baseline = pd.DataFrame(baseline_rows)
baseline_total_time = float(df_baseline["time_s"].sum())
print_table(df_baseline.round(4), "Separate baseline runs")
print("baseline_total_time:", baseline_total_time)


In [ ]:
# ============================================================
# Run CUDA C++ multi-EPS once
# ============================================================

OUT_LABELS = "multi_eps_labels.bin"
OUT_COUNTS = "multi_eps_neighbor_counts.bin"

cmd = [
    "./dbscan_multi_eps",
    INPUT_BIN,
    OUT_LABELS,
    str(N),
    str(D),
    str(float(EPS_VALUES[0])),
    str(float(EPS_VALUES[1])),
    str(float(EPS_VALUES[2])),
    str(MIN_SAMPLES),
    OUT_COUNTS,
]

start = time.time()
completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
multi_wall_time = time.time() - start

print(completed.stdout)

match = re.search(r"total=([0-9.]+) seconds", completed.stdout)
multi_cuda_event_time = float(match.group(1)) if match else np.nan

labels_flat = np.fromfile(OUT_LABELS, dtype=np.int32, count=len(EPS_VALUES) * N)
labels_multi = labels_flat.reshape((len(EPS_VALUES), N))

counts_flat = np.fromfile(OUT_COUNTS, dtype=np.int32, count=len(EPS_VALUES) * N)
counts_multi = counts_flat.reshape((len(EPS_VALUES), N))

rows = []
all_labels = {}

for e, (eps_name, eps_value) in enumerate(zip(EPS_NAMES, EPS_VALUES)):
    labels = relabel_consecutive(labels_multi[e])
    all_labels[f"multi_eps_{eps_name}"] = labels
    baseline_labels = baseline_labels_by_eps[eps_name]
    rows.append(evaluate(
        "cuda_cpp_multi_eps",
        eps_name,
        float(eps_value),
        labels,
        multi_wall_time,
        baseline_labels,
        {
            "cuda_event_total_s": multi_cuda_event_time,
            "baseline_total_time_s": baseline_total_time,
            "speedup_vs_all_baselines": baseline_total_time / multi_wall_time,
            "mean_neighbor_count_seen_until_core": float(np.mean(counts_multi[e])),
        },
    ))

df_multi = pd.DataFrame(rows)
print_table(df_multi.round(4), "Multi-EPS CUDA C++ results")


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_baseline["eps_name"], df_baseline["time_s"])
plt.ylabel("Time (s)")
plt.title("Separate baseline time per EPS")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(df_multi["eps_name"], df_multi["ARI_vs_baseline"])
plt.ylim(0, 1.05)
plt.ylabel("ARI vs corresponding baseline")
plt.title("Multi-EPS quality")
plt.tight_layout()
plt.show()

for eps_name, labels in baseline_labels_by_eps.items():
    plot_2d(X, labels, f"baseline - {eps_name}")

for name, labels in all_labels.items():
    plot_2d(X, labels, name)


## Interpretation

This is not just running DBSCAN three times. The CUDA binary computes the distance between points once inside `find_cores_multi_eps` and once inside `connect_cores_multi_eps`, then updates the structures for all EPS values. This mirrors the MultiK idea: more arithmetic work per data read.

Limitations: this is still a didactic brute-force DBSCAN (`O(n^2)`), without spatial indexes and without the drop-core approximation. The goal is to demonstrate the Multi-EPS strategy and compare quality/time against separate baseline runs.
